# ML Zoomcamp 2026 — Homework 9 (Serverless Deep Learning)
**Nicolau Calado Jofilsan** · 2026 cohort (DataTalks.Club)

**EN.** The hair classifier from module 8 is now a frozen ONNX model (an `.onnx` graph plus an external-data file, both
checked against published SHA-256 values). The homework runs it with ONNX Runtime on a CPU, reproduces the image
preprocessing, wraps everything in an AWS Lambda-compatible handler and invokes it as a Lambda container on the local
machine, so no cloud account is needed. The AWS deployment (ECR, Lambda, API Gateway) is optional and not graded.

**PT.** O classificador de cabelo do módulo 8 agora é um modelo ONNX congelado (um grafo `.onnx` mais um arquivo de dados
externo, ambos conferidos com SHA-256 publicados). O enunciado o executa com ONNX Runtime em CPU, reproduz o
pré-processamento da imagem, empacota tudo em um handler compatível com o AWS Lambda e o invoca como contêiner Lambda na
máquina local, sem precisar de conta na nuvem. A publicação na AWS (ECR, Lambda, API Gateway) é opcional e não vale nota.

Topics / Conteúdo: ONNX graph inspection, preprocessing as part of the model interface, Lambda handler, container image,
integrity checks for downloaded artifacts.

## 0. Setup and assets / Preparação e arquivos

In [1]:
import hashlib
import json
import os
import platform
import urllib.request

import numpy as np
import onnxruntime as ort
import PIL
from PIL import Image

print("python:", platform.python_version(), "| onnxruntime:", ort.__version__, "| numpy:", np.__version__,
      "| Pillow:", PIL.__version__)
if ort.__version__ != "1.23.2":
    print("NOTE: the reference setup pins onnxruntime 1.23.2 / a referência fixa o onnxruntime 1.23.2")

python: 3.13.17 | onnxruntime: 1.23.2 | numpy: 2.3.3 | Pillow: 11.3.0


**EN.** Every artifact is downloaded once and compared with the SHA-256 in the course's `asset_manifest.json` (copied
below). A file that does not match is not used. The ONNX file and its `.data` file must stay side by side with these exact
names. **PT.** Cada arquivo é baixado uma vez e comparado com o SHA-256 do `asset_manifest.json` do curso (copiado abaixo).
Um arquivo que não bate não é usado. O `.onnx` e o `.data` precisam ficar lado a lado, com esses nomes exatos.

In [2]:
PREFIX = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
SAMPLE_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
COURSE = ("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
          "main/cohorts/2026/homework/09-serverless/")

ASSETS = {  # file name -> (url, expected sha256)
    "hair_classifier_v1.onnx": (f"{PREFIX}/hair_classifier_v1.onnx",
                                "2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8"),
    "hair_classifier_v1.onnx.data": (f"{PREFIX}/hair_classifier_v1.onnx.data",
                                     "6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e"),
    "sample.jpeg": (SAMPLE_URL, "6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5"),
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, expected) in ASSETS.items():
    if not os.path.exists(name):
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response, open(name, "wb") as out:
            out.write(response.read())
    digest = sha256(name)
    assert digest == expected, f"checksum mismatch for {name}: {digest}"
    print(f"OK  {name:<32} {os.path.getsize(name):>12,} bytes  sha256 {digest[:16]}...")

# the three course files used below (handler, Dockerfile, smoke test) / os três arquivos do curso usados adiante
for name in ("lambda_function.py", "Dockerfile", "smoke_test.py", "requirements-lambda.txt"):
    if not os.path.exists(name):
        urllib.request.urlretrieve(COURSE + name, name)

OK  hair_classifier_v1.onnx                10,337 bytes  sha256 2b1adcb51745b736...


OK  hair_classifier_v1.onnx.data       80,355,328 bytes  sha256 6ba88582d6098a53...
OK  sample.jpeg                           398,272 bytes  sha256 6b3ae4003b5b6d54...


## Q1. Output node name / Nome do nó de saída

**EN.** Load the model with ONNX Runtime and read the graph interface. **PT.** Carregue o modelo com o ONNX Runtime e leia
a interface do grafo.

In [3]:
session = ort.InferenceSession("hair_classifier_v1.onnx", providers=["CPUExecutionProvider"])
for node in session.get_inputs():
    print("input :", node.name, node.shape, node.type)
for node in session.get_outputs():
    print("output:", node.name, node.shape, node.type)
print("\nQ1 =", session.get_outputs()[0].name)

input : input ['s77', 3, 200, 200] tensor(float)
output: output ['s77', 1] tensor(float)

Q1 = output


**Answer / Resposta Q1: `output`** — the graph has one input called `input` (batch × 3 × 200 × 200, `float32`) and one
output called `output` (batch × 1). The batch dimension prints as a symbol (such as `s77`) because it is dynamic; the
symbol's name has no meaning. The output already is a probability of the `straight` class, so no sigmoid is applied
afterwards. / o grafo tem uma entrada chamada `input` (lote × 3 × 200 × 200, `float32`) e uma saída chamada `output`
(lote × 1). A dimensão do lote aparece como um símbolo (como `s77`) porque é dinâmica; o nome do símbolo não tem
significado. A saída já é a probabilidade da classe `straight`, então nenhuma sigmoide é aplicada depois.

## Q2. Target size / Tamanho de destino

**EN.** The preprocessing is part of the model's interface: resize with bilinear interpolation, scale to 0–1, normalize
with the ImageNet mean and standard deviation, and reorder to channels-first. Using another interpolation or skipping the
RGB conversion changes the numbers. **PT.** O pré-processamento faz parte da interface do modelo: redimensionar com
interpolação bilinear, escalar para 0–1, normalizar com a média e o desvio-padrão do ImageNet e reordenar para canais
primeiro. Trocar a interpolação ou pular a conversão para RGB muda os números.

In [4]:
from io import BytesIO


def download_image(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


TARGET_SIZE = (200, 200)


def prepare_image(image):
    image = image.resize(TARGET_SIZE, Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]


print("Q2 =", f"{TARGET_SIZE[0]}x{TARGET_SIZE[1]}")

Q2 = 200x200


**Answer / Resposta Q2: `200x200`** — it matches the 200×200 input shape of the ONNX graph (Q1). / coincide com a forma de
entrada 200×200 do grafo ONNX (Q1).

## Q3. First normalized value / Primeiro valor normalizado

**EN.** The sample image is read from the checksummed local copy (the same bytes the course URL serves) and passed through
`prepare_image`; the question asks for the first value of the red channel. **PT.** A imagem de exemplo é lida da cópia
local já conferida por checksum (os mesmos bytes que a URL do curso entrega) e passa por `prepare_image`; a questão pede o
primeiro valor do canal vermelho.

In [5]:
image = Image.open("sample.jpeg").convert("RGB")
print("image size / tamanho da imagem:", image.size, "| mode:", image.mode)
tensor = prepare_image(image)
print("tensor shape:", tensor.shape, "| dtype:", tensor.dtype)
print("\nQ3 =", round(float(tensor[0, 0, 0, 0]), 3), f" (unrounded {float(tensor[0, 0, 0, 0]):.6f})")

image size / tamanho da imagem: (1024, 1024) | mode: RGB
tensor shape: (1, 3, 200, 200) | dtype: float32

Q3 = -1.056  (unrounded -1.056169)


**Answer / Resposta Q3: -1.056** — the grader accepts ±0.01. The value depends on the top-left pixel after
resizing and on `(x/255 − 0.485)/0.229`, so it is a quick check that the interpolation and normalization are right. /
o corretor aceita ±0,01. O valor depende do pixel do canto superior esquerdo depois do redimensionamento e de
`(x/255 − 0,485)/0,229`, então é uma verificação rápida de que a interpolação e a normalização estão corretas.

## Q4. Local ONNX inference / Inferência local com ONNX

In [6]:
output = session.run(["output"], {"input": tensor})[0]
probability = float(output.reshape(-1)[0])
print("output shape:", output.shape, "| dtype:", output.dtype)
print("\nQ4 =", round(probability, 3), f" (unrounded {probability:.6f})", "->", "straight" if probability >= 0.5 else "curly")

output shape: (1, 1) | dtype: float32

Q4 = 0.728  (unrounded 0.727697) -> straight


**Answer / Resposta Q4: 0.728** — probability that the hair is straight (`straight=1`). The grader accepts ±0.02.
/ probabilidade de o cabelo ser liso (`straight=1`). O corretor aceita ±0,02.

## Lambda handler / Handler do Lambda

**EN.** `lambda_function.py` is the course's handler: it loads the session once at import time (so warm invocations skip
the loading cost), downloads the image, prepares it, runs the model and returns an HTTP-style response. It accepts a direct
event or an API Gateway event whose `body` is a JSON string. Both are exercised below.
**PT.** `lambda_function.py` é o handler do curso: carrega a sessão uma vez na importação (assim as invocações "quentes"
não pagam o custo de carregar), baixa a imagem, prepara, roda o modelo e devolve uma resposta no estilo HTTP. Aceita um
evento direto ou um evento do API Gateway cujo `body` é uma string JSON. Os dois são testados abaixo.

In [7]:
import importlib
import sys
sys.path.insert(0, os.getcwd())
lambda_function = importlib.import_module("lambda_function")

# the handler downloads the image itself; here it reads the verified local copy through a file:// URL
# o handler baixa a imagem sozinho; aqui ele lê a cópia local verificada por uma URL file://
local_url = "file://" + os.path.abspath("sample.jpeg")
direct_event = {"image_url": local_url}
gateway_event = {"body": json.dumps({"image_url": local_url})}

for label, event in (("direct event / evento direto", direct_event), ("API Gateway-style event", gateway_event)):
    response = lambda_function.lambda_handler(event, None)
    print(label, "->", response["statusCode"], response["body"])

direct event / evento direto -> 200 {"straight_probability": 0.727697, "straight": true}


API Gateway-style event -> 200 {"straight_probability": 0.727697, "straight": true}


**EN.** Note that the handler rounds the probability to 6 decimals before returning it, so it can differ from the direct
ONNX call in the seventh decimal only. **PT.** Note que o handler arredonda a probabilidade para 6 casas antes de
devolvê-la, então ela pode diferir da chamada ONNX direta só na sétima casa.

## Q5. Lambda base image / Imagem base do Lambda

**EN.** The Dockerfile starts from the AWS Lambda base image, installs the pinned requirements into the task root and
copies the handler plus the checksummed model pair. **PT.** O Dockerfile parte da imagem base do AWS Lambda, instala os
requisitos fixados na raiz da tarefa e copia o handler mais o par de arquivos do modelo, já conferidos por checksum.

In [8]:
dockerfile = open("Dockerfile", encoding="utf-8").read()
print(dockerfile)
base = next(line.split()[1] for line in dockerfile.splitlines() if line.startswith("FROM"))
print("Q5 =", base.split("@")[0])

FROM public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be

COPY requirements-lambda.txt ${LAMBDA_TASK_ROOT}/
RUN python -m pip install --no-cache-dir -r ${LAMBDA_TASK_ROOT}/requirements-lambda.txt \
    --target ${LAMBDA_TASK_ROOT}

COPY lambda_function.py hair_classifier_v1.onnx hair_classifier_v1.onnx.data ${LAMBDA_TASK_ROOT}/

CMD ["lambda_function.lambda_handler"]

Q5 = public.ecr.aws/lambda/python:3.13


**Answer / Resposta Q5: `public.ecr.aws/lambda/python:3.13`** — the image is also pinned by an immutable `sha256` digest
after the `@`, so the same bytes are pulled every time; the tag alone could move. / a imagem também é fixada por um
digest `sha256` imutável depois do `@`, então os mesmos bytes são baixados sempre; só a tag poderia mudar.

## Q6. Invoke the container / Invocar o contêiner

**EN.** This step needs Docker on your machine. In a terminal, in the folder that holds the course files and the two model
files:
```
docker build -t mlzoomcamp-2026-serverless .
docker run --rm -p 9000:8080 mlzoomcamp-2026-serverless
```
and from a second terminal (on Windows PowerShell use `curl.exe`):
```
curl -s -XPOST "http://localhost:9000/2015-03-31/functions/function/invocations" -H "Content-Type: application/json" -d "{\"image_url\":\"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\"}"
```
Save the response to a file named `container_response.json` next to this notebook; the next cell reads it. If the file is
not there, the cell falls back to the handler call, which runs the same code, model and preprocessing but **not** inside the
container, and it says so.

**PT.** Este passo precisa do Docker na sua máquina. Em um terminal, na pasta que tem os arquivos do curso e os dois
arquivos do modelo, rode os dois comandos acima (`docker build` e `docker run`) e, de um segundo terminal (no PowerShell do
Windows use `curl.exe`), o comando `curl` acima. Salve a resposta em um arquivo `container_response.json` ao lado deste
notebook; a próxima célula o lê. Se o arquivo não existir, a célula usa a chamada do handler, que roda o mesmo código,
modelo e pré-processamento mas **não** dentro do contêiner, e avisa isso.

In [9]:
if os.path.exists("container_response.json"):
    response = json.load(open("container_response.json", encoding="utf-8"))
    source = "container response / resposta do contêiner"
else:
    response = lambda_function.lambda_handler(direct_event, None)
    source = "handler call, NOT the container / chamada do handler, NÃO o contêiner"

body = json.loads(response["body"])          # the body is a JSON *string* inside the response / o corpo é uma string JSON
print("source:", source)
print("status:", response["statusCode"], "| body:", body)
print("\nQ6 =", round(body["straight_probability"], 3))

source: container response / resposta do contêiner
status: 200 | body: {'straight_probability': 0.727697, 'straight': True}

Q6 = 0.728


**Answer / Resposta Q6: 0.728** — it must agree with Q4 (same model, image, preprocessing and CPU provider); the grader accepts
±0.02. The size shown by `docker images` is not graded. / deve concordar com a Q4 (mesmo modelo, imagem, pré-processamento
e provedor de CPU); o corretor aceita ±0,02. O tamanho mostrado por `docker images` não vale nota.

### Homework 9 — answers / respostas

| Question | Answer |
|---|---|
| Q1 Output node name | **`output`** |
| Q2 Target size | **`200x200`** |
| Q3 First value of the R channel (3 decimals) | **-1.056** |
| Q4 Local ONNX probability (3 decimals) | **0.728** |
| Q5 Runtime base image | **`public.ecr.aws/lambda/python:3.13`** |
| Q6 `straight_probability` from the container (3 decimals) | **0.728** |

### Notes / Observações

**EN.** 1) *Integrity.* The model pair and the sample image are verified by SHA-256 before use; the Dockerfile pins the base
image by digest and the requirements by exact version. 2) *Preprocessing is part of the model.* The same
`prepare_image` code appears in the handler, so training-time, notebook and container preprocessing cannot drift apart; the
2025 version of this homework left the interpolation implicit, and the first pixel and the output disagreed. On the sample image, resizing with `NEAREST` instead of bilinear moves the probability from 0.728 to 0.092 (and the first value from -1.056 to -1.073), which flips the predicted class. 3) *One image.*
A single probability says little about model quality: the model of module 8 reached about 0.72 accuracy on its held-out
images, so a confident output on one photo is not evidence of reliability. 4) *Image URL in the handler.* The handler
downloads whatever URL it receives; in a real service that is a request-forgery risk, and the URL should be restricted to
trusted hosts. 5) *Local container vs. AWS.* Running the image locally exercises the Lambda runtime interface; it does not
test cold-start time, memory limits or API Gateway.

**PT.** 1) *Integridade.* O par do modelo e a imagem de exemplo são verificados por SHA-256 antes do uso; o Dockerfile fixa a
imagem base por digest e os requisitos por versão exata. 2) *O pré-processamento faz parte do modelo.* O mesmo código de
`prepare_image` aparece no handler, então o pré-processamento do notebook e do contêiner não pode divergir; a versão de
2025 deste exercício deixava a interpolação implícita, e o primeiro pixel e a saída discordavam. Na imagem de exemplo, redimensionar com `NEAREST` em vez de bilinear move a probabilidade de 0,728 para 0,092 (e o primeiro valor de -1,056 para -1,073), o que inverte a classe prevista. 3) *Uma só imagem.* Uma
única probabilidade diz pouco sobre a qualidade do modelo: o modelo do módulo 8 chegou a cerca de 0,72 de acurácia nas
imagens de avaliação, então uma saída confiante em uma foto não é evidência de confiabilidade. 4) *URL de imagem no
handler.* O handler baixa qualquer URL que receber; num serviço real isso é um risco de falsificação de requisição (SSRF),
e a URL deveria ser restrita a hosts confiáveis. 5) *Contêiner local x AWS.* Rodar a imagem localmente exercita a interface
de execução do Lambda; não testa tempo de partida a frio, limites de memória nem o API Gateway.